In [ ]:
from pathlib import Path
import torch
import yaml

DATASET_DIR = Path(r"D:\Projects\cv_project\void_detection\data")
DATA_YAML = DATASET_DIR / "data.yaml"

print("Dataset YAML exists:", DATA_YAML.exists())

with open(DATA_YAML, "r") as f:
    data = yaml.safe_load(f)

print("\nClasses:", data.get("names"))
print("Number of classes:", data.get("nc"))

for split in ["train", "valid", "test"]:
    images = DATASET_DIR / split / "images"
    labels = DATASET_DIR / split / "labels"

    print(
        f"{split.upper()}: "
        f"{len(list(images.glob('*')))} image files, "
        f"{len(list(labels.glob('*.txt')))} label files"
    )

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import yaml

DATASET_DIR = Path(r"D:\Projects\cv_project\data")

with open(DATASET_DIR / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

# Point YOLO directly to the dataset folders
data["path"] = str(DATASET_DIR.resolve())
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

TRAIN_YAML = DATASET_DIR / "data_train.yaml"

with open(TRAIN_YAML, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Training YAML created:", TRAIN_YAML)
print(TRAIN_YAML.read_text())

In [ ]:
from pathlib import Path
import yaml

DATASET_DIR = Path(r"D:\Projects\cv_project\void_detection\data")

with open(DATASET_DIR / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

# Point YOLO directly to the dataset folders
data["path"] = str(DATASET_DIR.resolve())
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

TRAIN_YAML = DATASET_DIR / "data_train.yaml"

with open(TRAIN_YAML, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Training YAML created:", TRAIN_YAML)
print(TRAIN_YAML.read_text())

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolo11s.pt")

results = model.train(
    data=str(TRAIN_YAML),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=4,
    project="runs/void_detection",
    name="yolo11s_baseline",
    patience=10,
    plots=True,
    seed=42
)

In [ ]:
from ultralytics import YOLO

run_dir = Path(results.save_dir)
weights_path = run_dir / "weights" / "best.pt"

if not weights_path.is_file():
    raise FileNotFoundError(
        f"Best weights were not found at {weights_path}. "
        "Run the training cell and confirm it completed successfully."
    )

best_model = YOLO(str(weights_path))

metrics = best_model.val(
    data=str(TRAIN_YAML),
    split="test",
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu"
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:
best_model.predict(
    source=str(DATASET_DIR / "test" / "images"),
    imgsz=640,
    conf=0.25,
    save=True,
    device=0 if torch.cuda.is_available() else "cpu"
)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import random

DATASET_DIR = Path(r"D:\Projects\cv_project\data")
BEST_MODEL = Path(
    r"D:\Projects\cv_project\runs\void_detection\yolo11s_baseline\weights\best.pt"
)

# Load the trained model
if not weights_path.is_file():
    raise FileNotFoundError(f"Best weights not found: {weights_path}")

model = YOLO(str(weights_path))

# Pick a random test image
test_images = list((DATASET_DIR / "test" / "images").glob("*"))
test_images = [
    p for p in test_images
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
]

image_path = random.choice(test_images)

# Predict voids
result = next(iter(model.predict(
    source=str(image_path),
    imgsz=640,
    conf=0.25,
    device=0 if torch.cuda.is_available() else "cpu",
    verbose=False
)))

# Display annotated prediction
plt.figure(figsize=(12, 8))
plt.imshow(result.plot()[..., ::-1])
plt.title(f"Void Detection: {image_path.name}")
plt.axis("off")
plt.show()

boxes = getattr(result, "boxes", None)
print("Detections:", len(boxes) if boxes is not None else 0)

In [ ]:
from matplotlib.patches import Rectangle

if not test_images:
    raise FileNotFoundError("No test images were found.")

# Use the image selected in the previous cell, or fall back to the first test image.
if "image_path" not in globals():
    image_path = test_images[0]

image_path = Path(image_path)
label_path = DATASET_DIR / "test" / "labels" / f"{image_path.stem}.txt"

# Load ground-truth image and run prediction
image = plt.imread(image_path)
prediction = model.predict(
    source=str(image_path),
    imgsz=640,
    conf=0.25,
    device=0 if torch.cuda.is_available() else "cpu",
    verbose=False
)[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ground truth
axes[0].imshow(image)
axes[0].set_title("Actual voids (ground truth)")
height, width = image.shape[:2]

if label_path.is_file():
    for line in label_path.read_text().splitlines():
        values = line.split()
        if len(values) != 5:
            continue

        class_id, x_center, y_center, box_width, box_height = map(float, values)
        x = (x_center - box_width / 2) * width
        y = (y_center - box_height / 2) * height
        box_width *= width
        box_height *= height

        axes[0].add_patch(Rectangle(
            (x, y), box_width, box_height,
            fill=False, edgecolor="lime", linewidth=2
        ))
        class_name = model.names.get(int(class_id), str(int(class_id)))
        axes[0].text(x, y, class_name, color="black",
                     bbox=dict(facecolor="lime", alpha=0.8))
else:
    axes[0].text(0.5, 0.5, "Ground-truth label file not found",
                 transform=axes[0].transAxes, ha="center", color="red")

# Predictions
axes[1].imshow(prediction.plot()[..., ::-1])
axes[1].set_title("Predicted voids")

for ax in axes:
    ax.axis("off")

fig.suptitle(image_path.name)
plt.tight_layout()
plt.show()

print("Ground-truth labels:", label_path)
print("Predicted detections:", len(prediction.boxes) if prediction.boxes is not None else 0)